# LeWorldModel — Time-Linear Predictors (Colab)

Run this notebook on a **GPU runtime** (Runtime → Change runtime type → T4 GPU)
for the best performance.  All three predictor variants are tested:
1. **Baseline** — Transformer + softmax attention (O(T²))
2. **DeltaNet** — Linear attention via delta rule (O(T))
3. **Mamba** — State-space model with stateful rollouts (O(1) per step)

Optimized CUDA kernels are installed when a GPU is detected; pure-PyTorch fallbacks
are used otherwise.

In [ ]:
import sys, torch, platform
print(f"Python {platform.python_version()}")
print(f"PyTorch {torch.__version__}")
has_gpu = torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0) if has_gpu else 'NO GPU (using CPU)'}")

---
## 1. Clone the repository

In [ ]:
# Clone your fork — change USER/REPO to your own
!git clone https://github.com/YOUR_USER/YOUR_REPO.git le-wm
%cd le-wm

---
## 2. Install dependencies

In [ ]:
!pip install -q stable-pretraining einops

# Install optimized CUDA kernels (only if GPU is available)
if torch.cuda.is_available():
    !pip install -q causal-conv1d mamba-ssm

    # flash-linear-attention needs PyTorch >= 2.7 + Triton >= 3.3
    # Colab may ship older versions — try pip first, then GitHub
    ok = not __import__('subprocess').call(
        'pip install -q flash-linear-attention 2>/dev/null'.split()
    )
    if not ok:
        print("  pip install failed — trying GitHub source...")
        ok = not __import__('subprocess').call(
            'pip install -q git+https://github.com/fla-org/flash-linear-attention 2>/dev/null'.split()
        )
    if ok:
        print("  ✓ flash-linear-attention installed")
    else:
        print("  ✗ flash-linear-attention unavailable — using pure-PyTorch fallback")
else:
    print("No GPU detected — using pure-PyTorch fallbacks.")

print("\nBackend status:")
for pkg, name in [('mamba_ssm', 'Mamba SSM'), ('fla', 'FLA (DeltaNet)')]:
    try:
        __import__(pkg)
        print(f"  ✓ {name}: CUDA kernel active")
    except ImportError:
        print(f"  ✗ {name}: pure-PyTorch fallback")

---
## 3. Run verification tests

Validates forward/backward, rollout, and SIGReg for all 3 variants.

In [ ]:
!python test_models.py

---
## 4. Run scaling benchmark

Measures forward and rollout times across sequence lengths T=2..128.
This validates the O(T²) vs O(T) vs O(1) scaling curves.

In [ ]:
!python benchmark_scaling.py

---
## 5. Train all 3 variants (mini config)

Each model trains on PushT for 2 epochs at batch_size=8.
This is a smoke test — not enough for convergence, but enough to verify
the training loop works end-to-end.

In [ ]:
for model in ['lewm', 'lewm_deltanet', 'lewm_mamba']:
    print(f"\n{'='*60}")
    print(f"  Training {model}")
    print(f"{'='*60}")
    !python train.py model={model} data=pusht \
        trainer.max_epochs=2 \
        loader.batch_size=8 \
        wandb.enabled=False \
        2>&1 | tail -10

---
## 6. (Optional) Full training

Run a full training run (100 epochs) and monitor via WandB.
Update `wandb.entity` and `wandb.project` in `config/train/launcher/local.yaml` first.

In [ ]:
# Pick one:
# !python train.py model=lewm           data=pusht    # baseline
# !python train.py model=lewm_deltanet  data=pusht    # DeltaNet
# !python train.py model=lewm_mamba     data=pusht    # Mamba

---
## 7. (Optional) Evaluation

Requires a trained checkpoint.  See README for dataset setup.

In [ ]:
# !python eval.py --config-name=pusht.yaml policy=pusht/lewm

---
## Summary

| Variant | Complexity | Training | Rollout | Best for |
|---|---|---|---|---|
| Baseline (Transformer) | O(T²) per block | Fast at small T | Fast at small T | Short horizons ≤ 10 |
| DeltaNet | O(T) per block | Linear | Linear | Medium horizons |
| Mamba | O(T) fwd / O(1) step | Linear | Constant per step | Long horizons, real-time MPC |